# Customer Churn Analysis

dataset - Telco Customer Churn (got it from kaggle)

the main goal here is to figure out why customers are leaving the telecom company and which type of customers leave the most. also tried some ml models at the end to see if we can predict churn.

## importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

## loading the data

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)
df.head()

In [ ]:
# just checking basic stats
df.describe()

In [ ]:
df.info()

## Data Cleaning

found some issues when i ran df.info() -
- TotalCharges is object type but it should be float
- Churn column has yes/no, need to convert to 1/0 for calculations

In [ ]:
# TotalCharges has some spaces in it thats why its showing as object
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# checking how many nulls came after conversion
print(df['TotalCharges'].isnull().sum())

In [ ]:
# these 11 rows are customers with 0 tenure so just filling with 0
df['TotalCharges'].fillna(0, inplace=True)

# making a numeric version of churn for calculations
df['Churn_Binary'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('done')
print('total churned customers:', df['Churn_Binary'].sum())
print('churn rate:', round(df['Churn_Binary'].mean() * 100, 2), '%')

## EDA - Exploratory Data Analysis

now lets look at the data and find some patterns. ill make some charts to understand who is churning and why.

In [ ]:
# first just checking overall how many churned vs stayed
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))

colors = ['#4CAF50', '#E53935']
bars = ax.bar(churn_counts.index, churn_counts.values, color=colors, width=0.5, edgecolor='white')

for bar, pct in zip(bars, churn_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 40,
            f'{pct:.1f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Churn Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Churn')
ax.set_ylabel('Count')
ax.set_ylim(0, max(churn_counts.values) * 1.15)
sns.despine()
plt.tight_layout()
plt.show()

# roughly 26.5% customers have churned which is quite high

### does tenure affect churn?

tenure = how many months the person has been with the company. my guess is newer customers leave more.

In [ ]:
churned = df[df['Churn'] == 'Yes']['tenure']
stayed = df[df['Churn'] == 'No']['tenure']

fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(stayed, bins=30, alpha=0.6, color='#4CAF50', label='Stayed', edgecolor='white')
ax.hist(churned, bins=30, alpha=0.6, color='#E53935', label='Churned', edgecolor='white')

# adding avg lines so its easier to compare
ax.axvline(churned.mean(), color='#B71C1C', linestyle='--', linewidth=1.5,
           label=f'avg churned = {churned.mean():.0f} months')
ax.axvline(stayed.mean(), color='#1B5E20', linestyle='--', linewidth=1.5,
           label=f'avg stayed = {stayed.mean():.0f} months')

ax.set_title('Tenure: Churned vs Stayed', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Number of customers')
ax.legend(fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()

print(f'avg tenure churned: {churned.mean():.1f}')
print(f'avg tenure stayed: {stayed.mean():.1f}')

so yeah my guess was right, customers who leave have much lower tenure. the first few months are where most people decide if they want to stay or not.

### contract type vs churn

In [ ]:
contract_churn = df.groupby('Contract')['Churn_Binary'].mean().reset_index()
contract_churn.columns = ['Contract', 'Churn Rate']
contract_churn['Churn Rate'] = contract_churn['Churn Rate'] * 100
contract_churn = contract_churn.sort_values('Churn Rate', ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))

bar_colors = ['#E53935', '#FF8F00', '#4CAF50']
bars = ax.bar(contract_churn['Contract'], contract_churn['Churn Rate'],
              color=bar_colors, width=0.5, edgecolor='white')

for bar, val in zip(bars, contract_churn['Churn Rate']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, max(contract_churn['Churn Rate']) * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

this is a big difference. month to month customers are way more likely to leave, makes sense because they have no commitment. 2 year contract people almost never churn.

### monthly charges

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.boxplot(data=df, x='Churn', y='MonthlyCharges',
            palette={'No': '#4CAF50', 'Yes': '#E53935'},
            ax=ax, width=0.4)

ax.set_title('Monthly Charges vs Churn', fontsize=13, fontweight='bold')
ax.set_xlabel('Churned?')
ax.set_ylabel('Monthly Charges ($)')
sns.despine()
plt.tight_layout()
plt.show()

print('avg monthly charge (churned):', round(df[df['Churn']=='Yes']['MonthlyCharges'].mean(), 2))
print('avg monthly charge (stayed):', round(df[df['Churn']=='No']['MonthlyCharges'].mean(), 2))

### senior citizens

In [ ]:
df['SeniorCitizen_Label'] = df['SeniorCitizen'].map({1: 'Senior', 0: 'Non-Senior'})
senior_churn = df.groupby('SeniorCitizen_Label')['Churn_Binary'].mean() * 100

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(senior_churn.index, senior_churn.values,
              color=['#4CAF50', '#E53935'], width=0.4, edgecolor='white')

for bar, val in zip(bars, senior_churn.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Churn Rate: Senior vs Non-Senior', fontsize=13, fontweight='bold')
ax.set_xlabel('Customer Type')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, max(senior_churn.values) * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

### summary chart - all segments together

In [ ]:
summary = {
    'Month-to-Month Contract': df[df['Contract'] == 'Month-to-month']['Churn_Binary'].mean() * 100,
    'Tenure < 6 Months': df[df['tenure'] < 6]['Churn_Binary'].mean() * 100,
    'Senior Citizens': df[df['SeniorCitizen'] == 1]['Churn_Binary'].mean() * 100,
    'High Monthly Charges': df[df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)]['Churn_Binary'].mean() * 100,
    'Overall': df['Churn_Binary'].mean() * 100,
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Segment', 'Churn Rate (%)'])
summary_df = summary_df.sort_values('Churn Rate (%)', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))

colors = ['#5C6BC0' if s == 'Overall' else '#E53935' for s in summary_df['Segment']]
bars = ax.barh(summary_df['Segment'], summary_df['Churn Rate (%)'],
               color=colors, edgecolor='white', height=0.5)

for bar, val in zip(bars, summary_df['Churn Rate (%)']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.axvline(df['Churn_Binary'].mean() * 100, color='#1565C0', linestyle='--',
           linewidth=1.5, label=f'overall avg: {df["Churn_Binary"].mean()*100:.1f}%')

ax.set_title('Churn Rate by Segment', fontsize=13, fontweight='bold')
ax.set_xlabel('Churn Rate (%)')
ax.set_xlim(0, 65)
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()

---
## Machine Learning part

now i want to try and actually predict churn using ml models. i'll try 4 models and see which one does best.

first need to prepare the data - ml models only understand numbers so need to encode the text columns

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# working on a copy so the original df doesnt get messed up
df_ml = df.copy()

# dropping columns that are not needed
df_ml = df_ml.drop(['customerID', 'Churn', 'SeniorCitizen_Label'], axis=1)

# yes/no columns - converting to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df_ml[col] = df_ml[col].map({'Yes': 1, 'No': 0})

df_ml['gender'] = df_ml['gender'].map({'Male': 1, 'Female': 0})

# one hot encoding for columns with more than 2 options
df_ml = pd.get_dummies(df_ml, columns=['MultipleLines', 'InternetService',
                                        'OnlineSecurity', 'OnlineBackup',
                                        'DeviceProtection', 'TechSupport',
                                        'StreamingTV', 'StreamingMovies',
                                        'Contract', 'PaymentMethod'])

print(df_ml.shape)

In [ ]:
X = df_ml.drop('Churn_Binary', axis=1)
y = df_ml['Churn_Binary']

# 80-20 split, stratify so both sets have same churn ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('train size:', X_train.shape[0])
print('test size:', X_test.shape[0])

In [ ]:
# scaling - needed for logistic regression and knn
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)  # dont fit on test, only transform

### Model 1 - Logistic Regression

starting simple with logistic regression. even though name says regression its actually for classification.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_sc, y_train)

lr_pred = lr.predict(X_test_sc)

print('accuracy:', round(accuracy_score(y_test, lr_pred) * 100, 2), '%')
print()
print(classification_report(y_test, lr_pred, target_names=['Stayed', 'Churned']))

In [ ]:
# confusion matrix
# this shows how many were predicted correctly vs incorrectly
cm = confusion_matrix(y_test, lr_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'])
plt.title('Logistic Regression - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

### Model 2 - Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# keeping max depth low so it doesnt overfit
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print('accuracy:', round(accuracy_score(y_test, dt_pred) * 100, 2), '%')
print()
print(classification_report(y_test, dt_pred, target_names=['Stayed', 'Churned']))

In [ ]:
cm2 = confusion_matrix(y_test, dt_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'])
plt.title('Decision Tree - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

### Model 3 - Random Forest

random forest is basically many decision trees together. each tree votes and majority wins. usually performs better than single tree.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print('accuracy:', round(accuracy_score(y_test, rf_pred) * 100, 2), '%')
print()
print(classification_report(y_test, rf_pred, target_names=['Stayed', 'Churned']))

In [ ]:
cm3 = confusion_matrix(y_test, rf_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm3, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'])
plt.title('Random Forest - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# random forest gives feature importances which is really useful
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
top10 = feat_imp.sort_values(ascending=False)[:10]

fig, ax = plt.subplots(figsize=(9, 4))
top10.sort_values().plot(kind='barh', ax=ax, color='#1976D2', edgecolor='white')
ax.set_title('Top 10 features - Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance')
sns.despine()
plt.tight_layout()
plt.show()

# tenure and monthly charges are coming on top which matches the eda findings

### Model 4 - KNN

knn finds the k closest customers and predicts based on what they did. using k=5

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)

knn_pred = knn.predict(X_test_sc)

print('accuracy:', round(accuracy_score(y_test, knn_pred) * 100, 2), '%')
print()
print(classification_report(y_test, knn_pred, target_names=['Stayed', 'Churned']))

In [ ]:
cm4 = confusion_matrix(y_test, knn_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm4, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'])
plt.title('KNN - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

### comparing all models

lets see all 4 together. accuracy alone isnt the best metric here because the dataset is imbalanced (more non-churners). f1 score is better to look at.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

model_names = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'KNN']
all_preds = [lr_pred, dt_pred, rf_pred, knn_pred]

rows = []
for name, pred in zip(model_names, all_preds):
    rows.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, pred) * 100, 1),
        'Precision': round(precision_score(y_test, pred) * 100, 1),
        'Recall': round(recall_score(y_test, pred) * 100, 1),
        'F1': round(f1_score(y_test, pred) * 100, 1)
    })

results_df = pd.DataFrame(rows)
results_df

In [ ]:
x = np.arange(len(model_names))
w = 0.2

fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(x - 1.5*w, results_df['Accuracy'], w, label='Accuracy', color='#42A5F5', edgecolor='white')
ax.bar(x - 0.5*w, results_df['Precision'], w, label='Precision', color='#66BB6A', edgecolor='white')
ax.bar(x + 0.5*w, results_df['Recall'], w, label='Recall', color='#FFA726', edgecolor='white')
ax.bar(x + 1.5*w, results_df['F1'], w, label='F1 Score', color='#EF5350', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Score (%)')
ax.set_title('All Models Comparison', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 100)
sns.despine()
plt.tight_layout()
plt.show()

random forest wins. has the best accuracy and best f1 score. the feature importance it gave also matches what we found in eda - tenure and monthly charges are the biggest factors.

---

## Conclusion

from the analysis the main things i found -
- month to month contract customers have the highest churn rate
- new customers (less than 6 months) are most at risk
- higher monthly charges = more likely to leave
- senior citizens churn at almost double the rate

on ml side random forest gave the best results with highest accuracy and f1 score. logistic regression also did decent. knn and decision tree were a bit lower.

the company should focus on retaining month to month customers and new joiners in the first few months as those are the biggest risk groups.